In [14]:
from pathlib import Path
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from catboost import CatBoostClassifier
import numpy as np


# Load data
X_train = pd.read_csv(Path(r"data/X_train_selected.csv"))
X_test = pd.read_csv(Path(r"data/X_test_selected.csv"))
y_train = pd.read_csv(Path(r"data/y_train.csv")).squeeze("columns")
y_test = pd.read_csv(Path(r"data/y_test.csv")).squeeze("columns")


In [15]:
! uv pip install xgboost
! uv pip install lightgbm
! uv pip install catboost

Checked 1 package in 9ms
Checked 1 package in 8ms
Checked 1 package in 13ms


In [16]:
# Model
logreg_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

In [17]:
# Model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

In [18]:
# Model
xgb_model = XGBClassifier(
    n_estimators=450,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

In [19]:
#Model 
cat_boost = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="Logloss",
    random_seed=42,
    verbose=False
)

In [20]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

In [21]:
logreg_cv = cross_validate(
    logreg_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
logreg_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(logreg_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(logreg_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("Logistic Regression - Cross-Validation Results")
display(logreg_results)

Logistic Regression - Cross-Validation Results


,metric,mean,std
0,accuracy,0.816526,0.005188
1,precision,0.702078,0.007009
2,recall,0.719685,0.013537
3,f1,0.710737,0.009446


In [22]:
# CV
rf_cv = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
rf_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(rf_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(rf_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("Random Forest - Cross-Validation Results")
display(rf_results)

Random Forest - Cross-Validation Results


,metric,mean,std
0,accuracy,0.882867,0.004001
1,precision,0.933012,0.004765
2,recall,0.674542,0.013335
3,f1,0.782917,0.009125


In [23]:
# CV
cat_boost_cv = cross_validate(
    cat_boost,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
cat_boost_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(cat_boost_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(cat_boost_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("CatBoost - Cross-Validation Results")
display(cat_boost_results)

CatBoost - Cross-Validation Results


,metric,mean,std
0,accuracy,0.882718,0.006132
1,precision,0.925751,0.008673
2,recall,0.680212,0.019266
3,f1,0.784052,0.013532


In [24]:
# CV
xgb_cv = cross_validate(
    xgb_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Aggregate results
xgb_results = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [np.mean(xgb_cv[f"test_{m}"]) for m in scoring.keys()],
    "std": [np.std(xgb_cv[f"test_{m}"]) for m in scoring.keys()]
})

print("XGBoost - Cross-Validation Results")
display(xgb_results)

XGBoost - Cross-Validation Results


,metric,mean,std
0,accuracy,0.883755,0.007149
1,precision,0.912242,0.009165
2,recall,0.695811,0.019403
3,f1,0.789355,0.014739


## Versão Final fitted no total do train dataset

In [25]:
logreg_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
cat_boost.fit(X_train, y_train)

CatBoostClassifier(depth=6, eval_metric='Logloss', iterations=500, learning_rate=0.05, random_seed=42, verbose=False)

In [26]:
from pathlib import Path
import pandas as pd
import joblib

# Save fitted models
models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(logreg_model, models_dir / "logreg_model.joblib")
joblib.dump(rf_model, models_dir / "rf_model.joblib")
joblib.dump(xgb_model, models_dir / "xgb_model.joblib")
joblib.dump(cat_boost, models_dir / "cat_boost.joblib")

print("Models saved successfully.")

Models saved successfully.
